# Exp 03 · Architecture Deep-Dive (3): Prompt Template ↔ Tokenizer

**Goal**: see how the text instruction is wrapped in OpenVLA's fixed prompt template and split into tokens — and how it joins the 256 visual tokens into one sequence fed to Llama.

**No model, no GPU.** Tokenizer only, a few seconds on CPU. This is the last piece of Phase 2 — it stitches the action side and vision side into one complete input stream.

## The full input assembly
```
[256 visual tokens]  +  "In: What action should the robot take to {instruction}?\nOut:"
   └── from image ──┘     └──────────── text, split by the tokenizer ────────────────┘
                                            ↓ Llama reads this whole sequence
                                  autoregressively emits 7 action tokens -> decode to an action
```

- The wrapper text `In: What action...? Out:` is **always identical** — it's the cue that puts the model in 'action mode'.
- `Out:` is the handoff point: everything the model generates comes **after** it.
- Text uses **normal** tokens (not the vocab-tail 256 that actions reuse).

---
## Step 1 · Build the prompt and tokenize it

We reuse the same no-login Llama-2 tokenizer from notebook (1).

In [1]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained('NousResearch/Llama-2-7b-hf')

# OpenVLA's exact prompt template (the wrapper never changes; only {instruction} varies).
instruction = 'pick up the black bowl'
prompt = f'In: What action should the robot take to {instruction}?\nOut:'
print(repr(prompt))

/Users/duanxingjuan/GitPJ/openVLA-study/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'In: What action should the robot take to pick up the black bowl?\nOut:'


In [2]:
# Tokenize: turn the text into a list of token ids.
ids = tok(prompt).input_ids
print('number of text tokens:', len(ids))
print('token ids:', ids)
print()
# Show each token id next to the substring it represents.
print(f"{'#':>3}  {'id':>6}  piece")
for i, t in enumerate(ids):
    piece = tok.decode([t])
    print(f'{i:>3}  {t:>6}  {piece!r}')

number of text tokens: 20
token ids: [1, 512, 29901, 1724, 3158, 881, 278, 19964, 2125, 304, 5839, 701, 278, 4628, 12580, 29880, 29973, 13, 3744, 29901]

  #      id  piece
  0       1  '<s>'
  1     512  'In'
  2   29901  ':'
  3    1724  'What'
  4    3158  'action'
  5     881  'should'
  6     278  'the'
  7   19964  'robot'
  8    2125  'take'
  9     304  'to'
 10    5839  'pick'
 11     701  'up'
 12     278  'the'
 13    4628  'black'
 14   12580  'bow'
 15   29880  'l'
 16   29973  '?'
 17      13  '\n'
 18    3744  'Out'
 19   29901  ':'


### Read the token list
- **id `1`** at the front is the special **BOS** (begin-of-sequence) token Llama prepends automatically.
- The token ids are all **small numbers** (normal, frequently-used vocab) — totally different from the action tokens' `31744~31999` tail. Text and actions live in different parts of the same 32000-vocab.
- A single English word often splits into **multiple tokens** (sub-word pieces). Count: the wrapper + instruction together is well under 30 tokens.
- The last tokens spell out `Out:` — that's the cue after which the model starts generating action tokens.

---
## Step 2 · Find the `Out:` handoff point

Everything up to and including `Out:` is the **prompt** (the model reads it). What comes after is what the model **writes** — the 7 action tokens.

In [3]:
decoded = tok.decode(ids)
print('decoded back:', repr(decoded))
print('round-trip matches original (ignoring the auto BOS):',
      decoded.replace(tok.bos_token, '').strip() == prompt.strip())
print()

# Conceptually, at inference the full Llama input is:
#   [256 visual tokens]  ++  [these text tokens]  ->  model writes [7 action tokens]
N_VISUAL, N_ACTION = 256, 7
print(f'visual tokens : {N_VISUAL}')
print(f'text tokens   : {len(ids)}')
print(f'-> Llama reads : {N_VISUAL + len(ids)} tokens, then writes {N_ACTION} action tokens')

decoded back: '<s> In: What action should the robot take to pick up the black bowl?\nOut:'
round-trip matches original (ignoring the auto BOS): True

visual tokens : 256
text tokens   : 20
-> Llama reads : 276 tokens, then writes 7 action tokens


## ✅ Section summary — Phase 2 complete
- [x] Prompt template `In: What action should the robot take to {instruction}?\nOut:` is fixed; only `{instruction}` varies
- [x] Text tokenizes into **normal** (low-id) tokens; actions reuse the **tail** (31744–31999) — same vocab, different regions
- [x] Full input = `[256 visual tokens] ++ [text tokens]`; `Out:` is the handoff after which the model writes 7 action tokens

### The whole pipeline, end to end
```
image ─→ SigLIP+DINOv2 ─→ 256 visual tokens ┐
                                             ├─→ Llama-2 ─→ 7 action tokens ─→ decode (256-bin) ─→ robot action
instruction ─→ prompt template ─→ text tokens ┘
```

**Phase 2 done.** Next: **Phase 3 — LoRA finetuning** (needs a GPU; plan: 1× A100 80GB on RunPod).